# Sprint 2 — Pipelines, muestra estratificada y balanceo

    El target está desbalanceado (~12.3% clase 1). Como el FP es más costoso, **no aplicamos SMOTE por defecto**: puede aumentar falsos positivos. Se deja como variante opcional para comparar.

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.io_utils import load_kick_data, save_df
from src.preprocessing import prepare_features, split_X_y

pd.set_option('display.max_columns', 120)
print('Proyecto:', PROJECT_ROOT)

from sklearn.model_selection import train_test_split
from src.preprocessing import build_preprocessor, get_column_groups
import joblib

Proyecto: /Users/alexandralozano/dp261-g1


In [6]:
train_full = pd.read_csv(PROCESSED_DIR / 'train_full.csv')
X_train_full, y_train_full = split_X_y(train_full)

if len(train_full) > SAMPLE_SIZE:
    X_sample, _, y_sample, _ = train_test_split(
        X_train_full, y_train_full,
        train_size=SAMPLE_SIZE,
        stratify=y_train_full,
        random_state=RANDOM_STATE
    )
else:
    X_sample, y_sample = X_train_full, y_train_full

train_sample = X_sample.copy(); train_sample[TARGET] = y_sample.values
save_df(train_sample, PROCESSED_DIR / 'train_sample.csv')
print('train_sample:', train_sample.shape)
print(y_sample.value_counts(normalize=True))

train_sample: (20000, 41)
IsBadBuy
0    0.877
1    0.123
Name: proportion, dtype: float64


In [7]:
numeric_cols, categorical_cols = get_column_groups(X_sample)
print('Numéricas:', len(numeric_cols))
print('Categóricas:', len(categorical_cols))
print('Categóricas:', categorical_cols)

Numéricas: 26
Categóricas: 14
Categóricas: ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST']


In [8]:
preprocessors = {
    'linear': build_preprocessor(X_sample, mode='linear'),
    'tree_ohe': build_preprocessor(X_sample, mode='tree_ohe'),
    'ordinal': build_preprocessor(X_sample, mode='ordinal'),
}
MODELS_DIR.mkdir(exist_ok=True)
for name, preproc in preprocessors.items():
    joblib.dump(preproc, MODELS_DIR / f'preprocessor_{name}.pkl')
print('Preprocesadores guardados en models/')

Preprocesadores guardados en models/


## Cuándo considerar balanceo

- Si el modelo no detecta casi ningún positivo: probar `class_weight` o SMOTE.
- Si precision cae mucho: evitar SMOTE y subir threshold.
- Para XGBoost/LightGBM: preferir `scale_pos_weight` antes que SMOTE.
- El test final nunca se balancea.